# Federated Learning for Privacy-Preserving Medical Diagnosis -- Colab Training

Real-data run of the **core architecture** on NIH ChestX-ray14 (GPU):
centralised ceiling + single-site floor + **federated FedAvg** global model.
Defence layers (DP, robust aggregation, attacks) are OFF -- this is the core
floor / federated / ceiling comparison (Tasks 4, 5, 6).

**Runtime:** ~1-2 h on a Colab T4. Set the runtime to **GPU** first:
`Runtime -> Change runtime type -> T4 GPU`.

**What it does, end to end:**
1. install the federated/DP stack
2. clone this repo
3. download NIH ChestX-ray14 from Kaggle: label CSVs (official) + all images from a
   224x224 resized mirror (~4 GB, not the full 45 GB)
4. run the core suite on a patient-aware ~20k-image subset
5. render charts + a short report, and copy everything to your Google Drive


## 1. Check the GPU

In [ ]:
import torch, sys
print("Python:", sys.version.split()[0])
print("torch :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

## 2. Install dependencies

torch / torchvision are preinstalled on Colab -- we do **not** reinstall them. We add
Flower **core** (no Ray; the in-process orchestrator is used), Opacus, and a few
utilities. Takes ~1 min.

In [ ]:
!pip install -q "flwr>=1.8,<2.0" "opacus>=1.4" torchmetrics scikit-learn pyyaml typer pandas pillow
print("deps installed")

## 3. Clone the project repo

Make sure the latest code is pushed to GitHub first (including the
`src/experiments/data_setup.py` fix). If the repo is private, replace the URL with a
token form: `https://<TOKEN>@github.com/<user>/<repo>.git`.

In [ ]:
import os
REPO_URL = "https://github.com/Pooja-Goel07/Federated-Learning-for-Privacy-Preserving-Medical-Diagnosis.git"
if not os.path.exists("/content/fl"):
    !git clone -q $REPO_URL /content/fl
%cd /content/fl
!git log --oneline -1

## 4. Kaggle credentials

The dataset comes from Kaggle, which needs an API token:
1. kaggle.com -> your profile -> **Settings** -> **API** -> **Create New Token**.
2. This downloads `kaggle.json`.
3. Run the cell below and upload that file when prompted.

In [ ]:
from google.colab import files
import os, pathlib
up = files.upload()                       # choose kaggle.json
pathlib.Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
!pip install -q --upgrade kaggle          # 2.2.x fixes per-file download bugs
print("kaggle configured")

## 5. Download NIH ChestX-ray14

The official dataset stores the 112k images as *folders* (not downloadable zips) and is
45 GB. So we take the two label CSVs from the official dataset (small, per-file download
works) and all the **images from a 224x224 resized mirror** (one ~4 GB download -- and we
train at 224px anyway, so no quality is lost and every bounding-box/severity image is
present).

The last line checks that on-disk image filenames match `Data_Entry_2017.csv`; that
number should be ~112k. If it is 0, the mirror used different filenames -- see the
fallback note below.

In [ ]:
import os, glob, zipfile
DATA = "/content/nih"
os.makedirs(DATA, exist_ok=True)
OFFICIAL = "nih-chest-xrays/data"
RESIZED  = "khanfashee/nih-chest-x-ray-14-224x224-resized"   # all images, 224px, ~4 GB

# 1) label + bounding-box CSVs from the OFFICIAL dataset (per-file download works for CSVs)
for f in ["Data_Entry_2017.csv", "BBox_List_2017.csv"]:
    if not os.path.exists(f"{DATA}/{f}"):
        !kaggle datasets download -d $OFFICIAL -f "$f" -p $DATA
        zp = f"{DATA}/{f}.zip"                 # Kaggle delivers some CSVs zipped
        if os.path.exists(zp):
            with zipfile.ZipFile(zp) as z: z.extractall(DATA)
            os.remove(zp)

# 2) all images at 224px from the resized mirror (downloads + unzips in one step)
if not glob.glob(f"{DATA}/**/*.png", recursive=True):
    !kaggle datasets download -d $RESIZED -p $DATA --unzip

# sanity: how many on-disk images line up with the label file
import pandas as pd
idx   = set(pd.read_csv(f"{DATA}/Data_Entry_2017.csv")["Image Index"])
names = {os.path.basename(p) for p in glob.glob(f"{DATA}/**/*.png", recursive=True)}
print("CSVs present:", os.path.exists(f"{DATA}/Data_Entry_2017.csv"),
      os.path.exists(f"{DATA}/BBox_List_2017.csv"))
print("images on disk:", len(names), "| matching Data_Entry:", len(idx & names))

**Fallback if the matching count is 0 or the mirror is unavailable:** use the official
*sample* (5,606 images, ~1 GB) instead -- replace the image download line above with:

    !kaggle datasets download -d nih-chest-xrays/sample -p /content/nih --unzip

It is smaller (so fewer severity images), but the rest of the notebook is unchanged.

## 6. Configure the run

Load `configs/colab_real.yaml` and point it at the Colab paths. Tweak
`subsample.max_images`, `training.local_epochs`, or `training.rounds` to trade runtime
for quality.

In [ ]:
import sys; sys.path.insert(0, "/content/fl")
from src.experiments.config import load_config

cfg = load_config("configs/colab_real.yaml")
cfg["data"]["root"] = DATA
cfg["partition"]["manifest_path"] = f"{DATA}/manifest.json"
cfg["data"]["subsample"]["max_images"] = 20000   # raise for a bigger run
# Baseline epochs are matched to the number of rounds so centralised is a FAIR ceiling
# (federation does ~rounds x local_epochs passes over the data). If these differ, the
# centralised number is a lightly-trained reference, not a strict upper bound.
cfg["training"]["rounds"] = 10                   # federated rounds
cfg["training"]["local_epochs"] = cfg["training"]["rounds"]   # centralised + single-site epochs
print("image_size:", cfg["data"]["image_size"], "| subsample:", cfg["data"]["subsample"])

## 7. Run the core suite

Centralised -> single-site -> federated, all on the same partition and test set.
Progress prints per epoch / per round. This is the long cell (~1-2 h).

In [ ]:
from src.experiments.suite import run_core_suite

OUT = "/content/fl/results/colab"
summary = run_core_suite(
    cfg, out_dir=OUT, data_root=DATA, fed_local_epochs=1,
    extra_setup={"dataset": "NIH ChestX-ray14 (224px, subset)",
                 "image_size": cfg["data"]["image_size"],
                 "baseline_epochs": cfg["training"]["local_epochs"]},
)

## 8. Charts + report

In [ ]:
from src.evaluation.report import generate_report
generate_report(OUT)

from IPython.display import Image, display, Markdown
for p in ["03_floor_fed_ceiling", "01_learning_curve", "02_global_loss", "04_per_pathology_auc"]:
    display(Image(f"{OUT}/plots/{p}.png"))
display(Markdown(open(f"{OUT}/REPORT.md").read()))

## 9. Save results to Google Drive

Copies results (JSON + plots + report) to your Drive so they survive the runtime
shutting down. Download them and commit into `results/` in the repo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil, os
dst = "/content/drive/MyDrive/fl-medical-results/colab"
os.makedirs(os.path.dirname(dst), exist_ok=True)
if os.path.exists(dst): shutil.rmtree(dst)
shutil.copytree(OUT, dst)
print("copied results ->", dst)